In [ ]:
# !pip install --upgrade matplotlib opencv-python

In [ ]:
# !pip uninstall matplotlib -y
# !pip install matplotlib

In [1]:
%matplotlib tkagg

In [2]:
import os
import numpy as np
import matplotlib.pyplot as plt

BASE_DIR       = r"D:\FCAI\plain_coupon\infared_data"
REFERENCE_CASE = "cr_500_phi_09"
OUTPUT_SUBDIR  = "time_average"
INPUT_CSV      = "time_average.csv"
HEADER_LINES   = 9
CMAP           = "inferno"

def read_csv(filepath):
    with open(filepath, "r") as f:
        lines = f.readlines()
    data_lines = [l.strip() for l in lines[HEADER_LINES + 1:] if l.strip()]
    return np.array([[float(v) for v in row.split(",")] for row in data_lines], dtype=np.float64)

ref_csv  = os.path.join(BASE_DIR, REFERENCE_CASE, OUTPUT_SUBDIR, INPUT_CSV)
ref_data = read_csv(ref_csv)
n_rows, n_cols = ref_data.shape

# Store points here — accessible from any cell after clicking
src_pts = []

fig, ax = plt.subplots(figsize=(13, 8))
ax.imshow(ref_data, cmap=CMAP, aspect="auto", origin="upper", interpolation="nearest")
ax.set_xticks(np.arange(0, n_cols, 20))
ax.set_yticks(np.arange(0, n_rows, 20))
ax.tick_params(labelsize=7)
ax.grid(color="white", linewidth=0.3, alpha=0.4)
ax.set_title(
    f"{REFERENCE_CASE}  ({n_rows} rows x {n_cols} cols)\n"
    "Click 4 corners:  top-left → top-right → bottom-right → bottom-left",
    fontsize=11
)

COLORS = ["cyan", "lime", "orange", "magenta"]
LABELS = ["1-TL", "2-TR", "3-BR", "4-BL"]

def on_click(event):
    if event.inaxes is not ax or event.button != 1 or len(src_pts) >= 4:
        return
    col, row = event.xdata, event.ydata
    idx = len(src_pts)
    src_pts.append((col, row))
    ax.plot(col, row, "o", color=COLORS[idx], markersize=10,
            markeredgecolor="white", markeredgewidth=1.5, zorder=5)
    ax.text(col + 4, row - 4, LABELS[idx], color=COLORS[idx],
            fontsize=11, fontweight="bold", zorder=6)
    fig.canvas.draw_idle()
    # Write directly to figure title so you can see it in the popup window
    current = "\n".join([f"  {LABELS[i]}: col={src_pts[i][0]:.1f}, row={src_pts[i][1]:.1f}"
                         for i in range(len(src_pts))])
    ax.set_title(
        f"Points collected ({len(src_pts)}/4):\n{current}",
        fontsize=10
    )
    if len(src_pts) == 4:
        xs = [p[0] for p in src_pts] + [src_pts[0][0]]
        ys = [p[1] for p in src_pts] + [src_pts[0][1]]
        ax.plot(xs, ys, "--", color="white", linewidth=1.5, alpha=0.7, zorder=4)
    fig.canvas.draw_idle()

fig.canvas.mpl_connect("button_press_event", on_click)
plt.tight_layout()
plt.show()

In [3]:
# Verify points were captured
print("Collected points:", src_pts)

Collected points: [(194.76034966834075, 68.94717927696746), (309.05693016700405, 67.55156076062485), (310.10392174409105, 181.9922791007196), (195.80734124542775, 182.55052650725662)]


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import cv2

BASE_DIR        = r"D:\FCAI\plain_coupon\infared_data"
REFERENCE_CASE  = "cr_500_phi_09"
OUTPUT_SUBDIR   = "time_average"
INPUT_CSV       = "time_average.csv"
OUTPUT_CSV      = "corrected.csv"
PLOT_SUBDIR     = "time_averaged_plot_intensity"
PLOT_FORMAT     = "png"
DPI             = 150
CMAP            = "inferno"
HEADER_LINES    = 9
SKIP_FOLDERS    = {"time_average", "time_averaged_plot_intensity"}

def read_csv(filepath):
    with open(filepath, "r") as f:
        lines = f.readlines()
    header     = lines[:HEADER_LINES]
    data_lines = [l.strip() for l in lines[HEADER_LINES + 1:] if l.strip()]
    data = np.array([[float(v) for v in row.split(",")] for row in data_lines], dtype=np.float64)
    return header, data

def write_csv(out_path, header_lines, data, note="PERSPECTIVE_CORRECTED"):
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "w", newline="") as f:
        for line in header_lines:
            stripped = line.rstrip("\n").rstrip("\r")
            if stripped.startswith("Filename"):
                f.write(f"{stripped}  [{note}]\n")
            else:
                f.write(line if line.endswith("\n") else line + "\n")
        f.write("\n")
        for row in data:
            f.write(",".join(f"{v:.6e}" for v in row) + "\n")

def compute_output_size(src_pts):
    tl, tr, br, bl = [np.array(p) for p in src_pts]
    width  = int(round(max(np.linalg.norm(tr - tl), np.linalg.norm(br - bl))))
    height = int(round(max(np.linalg.norm(bl - tl), np.linalg.norm(br - tr))))
    return width, height

def correct_perspective(data, src_pts, out_w, out_h):
    dst_pts = np.array([[0, 0], [out_w-1, 0], [out_w-1, out_h-1], [0, out_h-1]], dtype=np.float32)
    src     = np.array(src_pts, dtype=np.float32)
    H, _    = cv2.findHomography(src, dst_pts)
    warped  = cv2.warpPerspective(data.astype(np.float32), H, (out_w, out_h),
                                  flags=cv2.INTER_LINEAR,
                                  borderMode=cv2.BORDER_CONSTANT, borderValue=0.0)
    return warped.astype(np.float64)

def apply_correction(src_pts):
    if len(src_pts) != 4:
        print(f"ERROR: need exactly 4 points, got {len(src_pts)}.")
        return

    base = os.path.normpath(BASE_DIR)
    ref_csv = os.path.join(base, REFERENCE_CASE, OUTPUT_SUBDIR, INPUT_CSV)
    ref_header, ref_data = read_csv(ref_csv)
    out_w, out_h = compute_output_size(src_pts)

    print("=" * 62)
    print("  Source points:")
    for lbl, pt in zip(["top-left","top-right","bottom-right","bottom-left"], src_pts):
        print(f"    {lbl:15s}:  col={pt[0]:.1f},  row={pt[1]:.1f}")
    print(f"  Output size : {out_h} rows x {out_w} cols")

    # Preview on reference case
    ref_corr = correct_perspective(ref_data, src_pts, out_w, out_h)
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    axes[0].imshow(ref_data, cmap=CMAP, aspect="auto", origin="upper", interpolation="nearest")
    xs = [p[0] for p in src_pts] + [src_pts[0][0]]
    ys = [p[1] for p in src_pts] + [src_pts[0][1]]
    axes[0].plot(xs, ys, "c--o", markersize=6, linewidth=1.5)
    axes[0].set_title(f"Original  |  {ref_data.shape[0]} x {ref_data.shape[1]}")
    axes[1].imshow(ref_corr, cmap=CMAP, aspect="auto", origin="upper", interpolation="nearest")
    axes[1].set_title(f"Corrected  |  {ref_corr.shape[0]} x {ref_corr.shape[1]}")
    fig.suptitle(f"Preview — {REFERENCE_CASE}", fontsize=12)
    plt.tight_layout()
    plt.show()

    ans = input("\nLooks correct? Apply to ALL cases? [y/n]: ").strip().lower()
    if ans != "y":
        print("Cancelled. Adjust src_pts and call apply_correction() again.")
        return

    # Batch process all cases
    entries   = sorted(os.listdir(base))
    case_dirs = [os.path.join(base, e) for e in entries
                 if os.path.isdir(os.path.join(base, e)) and e not in SKIP_FOLDERS]

    print(f"\n  Processing {len(case_dirs)} case(s)...\n")
    ok_list, skip_list = [], []

    for case_dir in case_dirs:
        case_name = os.path.basename(case_dir)
        avg_csv   = os.path.join(case_dir, OUTPUT_SUBDIR, INPUT_CSV)
        if not os.path.isfile(avg_csv):
            print(f"  [SKIP] {case_name}  — time_average.csv not found")
            skip_list.append(case_name)
            continue

        header, data = read_csv(avg_csv)
        corrected    = correct_perspective(data, src_pts, out_w, out_h)

        # Save corrected CSV
        out_csv = os.path.join(case_dir, OUTPUT_SUBDIR, OUTPUT_CSV)
        write_csv(out_csv, header, corrected)

        # Save corrected PNG
        out_png = os.path.join(case_dir, PLOT_SUBDIR, f"{case_name}_corrected.{PLOT_FORMAT}")
        os.makedirs(os.path.dirname(out_png), exist_ok=True)
        fig, ax = plt.subplots(figsize=(10, 6))
        ax.imshow(corrected, cmap=CMAP, aspect="auto", origin="upper", interpolation="nearest")
        plt.colorbar(ax.images[0], ax=ax).set_label("Counts")
        ax.set_title(f"Corrected  |  {case_name}\n{corrected.shape[0]} rows x {corrected.shape[1]} cols")
        plt.tight_layout()
        fig.savefig(out_png, dpi=DPI, bbox_inches="tight")
        plt.close(fig)

        print(f"  [OK]  {case_name}  →  {corrected.shape[0]} rows x {corrected.shape[1]} cols")
        ok_list.append(case_name)

    print(f"\n  Done.  Processed: {len(ok_list)}   Skipped: {len(skip_list)}")
    if skip_list:
        for s in skip_list:
            print(f"    x  {s}")

print("apply_correction() is ready.")

In [ ]:
# !pip install opencv-python

In [ ]:
apply_correction(src_pts)